# Learning 01: GLiNER — Zero-Shot NER Basics

**GLiNER** (Generalist and Lightweight Named Entity Recognition) is a transformer-based model that
extracts entities of *any* type specified at runtime — no retraining needed.

Instead of a fixed label set (PERSON / ORG / LOC), you pass a list of entity descriptions:
`["person", "organization", "award", "spacecraft"]` and the model finds spans that match.

**`gliner-bi-edge-v2.0`** is the bi-encoder variant:
- **Text Encoder** (ModernBERT-based) processes the document
- **Label Encoder** (MiniLM) embeds entity type descriptions independently
- The two encoders never see each other → label embeddings can be **pre-computed once** and reused
- Result: ~130× faster than uni-encoder when working with 1000+ entity types

## Setup

In [1]:
from gliner import GLiNER
import json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "news_articles.json")) as f:
    articles = json.load(f)

print(f"✓ Loaded {len(articles)} articles")
print(f"  Sample: {articles[0]['text'][:80]}...")

✓ Loaded 10 articles
  Sample: Elon Musk announced that Tesla will open a new Gigafactory in Austin, Texas on M...


## 1. Load the Model

The model downloads from HuggingFace on first use (~120 MB for edge).

In [2]:
model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")
print("✓ Model loaded")

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/gliner/model.py:447: UserWarning: Resizing embeddings is not supported for bi-encoder models.
  instance.resize_embeddings()


✓ Model loaded


## 2. Basic `predict_entities`

```python
entities = model.predict_entities(text, labels, threshold=0.3)
```

Each entity is a dict:
```python
{"text": "Elon Musk", "label": "person", "start": 0, "end": 9, "score": 0.94}
```

- `threshold` — minimum confidence score (0.3 is a good starting point)
- Lower threshold → higher recall, more false positives
- Higher threshold → higher precision, fewer false positives

In [3]:
text = articles[0]["text"]
labels = ["person", "organization", "location", "date"]

entities = model.predict_entities(text, labels, threshold=0.3)

print(f"Text: {text}\n")
print(f"Found {len(entities)} entities:")
for e in entities:
    print(f"  [{e['label']:15}] {e['text']!r:35} score={e['score']:.2f}")

Text: Elon Musk announced that Tesla will open a new Gigafactory in Austin, Texas on March 22, 2024, creating over 5,000 manufacturing jobs and boosting the local economy.

Found 6 entities:
  [person         ] 'Elon Musk'                         score=0.73
  [organization   ] 'Tesla'                             score=0.81
  [organization   ] 'Gigafactory'                       score=0.42
  [location       ] 'Austin'                            score=0.49
  [location       ] 'Texas'                             score=0.62
  [date           ] 'March 22, 2024'                    score=0.82


## 3. Entity Output Format

The `start` and `end` fields are character-level offsets into the original text.
This is useful for precise highlighting or replacement.

In [4]:
# Verify offsets
for e in entities:
    extracted = text[e["start"]:e["end"]]
    match = "✓" if extracted == e["text"] else "✗"
    print(f"  {match} [{e['start']}:{e['end']}] text[start:end]={extracted!r}  entity['text']={e['text']!r}")

  ✓ [0:9] text[start:end]='Elon Musk'  entity['text']='Elon Musk'
  ✓ [25:30] text[start:end]='Tesla'  entity['text']='Tesla'
  ✓ [47:58] text[start:end]='Gigafactory'  entity['text']='Gigafactory'
  ✓ [62:68] text[start:end]='Austin'  entity['text']='Austin'
  ✓ [70:75] text[start:end]='Texas'  entity['text']='Texas'
  ✓ [79:93] text[start:end]='March 22, 2024'  entity['text']='March 22, 2024'


## 4. Using Domain-Specific Labels

GLiNER is zero-shot — you can use any English description as a label.
The label encoder embeds your description and matches it against token spans.

In [5]:
# Science domain
science_text = articles[3]["text"]  # CERN article
science_labels = ["research institution", "location", "date", "scientific discovery", "university"]

entities = model.predict_entities(science_text, science_labels, threshold=0.3)
print(f"Science article entities:")
for e in entities:
    print(f"  [{e['label']:25}] {e['text']!r}")

Science article entities:
  [research institution     ] 'CERN'
  [location                 ] 'Geneva'
  [location                 ] 'Switzerland'
  [date                     ] 'July 4, 2012'
  [research institution     ] 'MIT'
  [research institution     ] 'Stanford University'


In [6]:
# Finance domain
finance_text = articles[4]["text"]  # JPMorgan article
finance_labels = ["person", "company", "government body", "location", "date"]

entities = model.predict_entities(finance_text, finance_labels, threshold=0.3)
print(f"Finance article entities:")
for e in entities:
    print(f"  [{e['label']:20}] {e['text']!r}")

Finance article entities:
  [person              ] 'Jamie Dimon'
  [company             ] 'JPMorgan Chase'
  [government body     ] 'Senate Banking Committee'
  [location            ] 'Washington D.C.'
  [date                ] 'September 22, 2023'


## 5. Threshold Tuning

The right threshold depends on your precision/recall tradeoff.

In [7]:
text = articles[5]["text"]  # OpenAI article
labels = ["person", "organization", "location", "date"]

for threshold in [0.1, 0.3, 0.5, 0.7]:
    entities = model.predict_entities(text, labels, threshold=threshold)
    print(f"  threshold={threshold}: {len(entities)} entities found")
    for e in entities:
        print(f"    [{e['label']:12}] {e['text']!r:30} score={e['score']:.2f}")
    print()

  threshold=0.1: 8 entities found
    [person      ] 'Sam Altman'                   score=0.75
    [organization] 'CEO'                          score=0.14
    [organization] 'OpenAI'                       score=0.81
    [location    ] 'San Francisco'                score=0.66
    [date        ] 'November 22, 2023'            score=0.78
    [organization] 'board'                        score=0.33
    [organization] 'Microsoft'                    score=0.84
    [organization] 'company'                      score=0.21



  threshold=0.3: 6 entities found
    [person      ] 'Sam Altman'                   score=0.75
    [organization] 'OpenAI'                       score=0.81
    [location    ] 'San Francisco'                score=0.66
    [date        ] 'November 22, 2023'            score=0.78
    [organization] 'board'                        score=0.33
    [organization] 'Microsoft'                    score=0.84



  threshold=0.5: 5 entities found
    [person      ] 'Sam Altman'                   score=0.75
    [organization] 'OpenAI'                       score=0.81
    [location    ] 'San Francisco'                score=0.66
    [date        ] 'November 22, 2023'            score=0.78
    [organization] 'Microsoft'                    score=0.84



  threshold=0.7: 4 entities found
    [person      ] 'Sam Altman'                   score=0.75
    [organization] 'OpenAI'                       score=0.81
    [date        ] 'November 22, 2023'            score=0.78
    [organization] 'Microsoft'                    score=0.84



## 6. Batch Processing with `.inference()`

For multiple documents, use `model.inference()` which processes them in batches internally.

In [8]:
texts = [a["text"] for a in articles]
labels = ["person", "organization", "location", "date"]

# model.inference() processes all texts at once
all_results = model.inference(texts, labels, threshold=0.3, batch_size=4)

print(f"Processed {len(all_results)} articles")
for i, (article, entities) in enumerate(zip(articles, all_results)):
    persons = [e['text'] for e in entities if e['label'] == 'person']
    orgs = [e['text'] for e in entities if e['label'] == 'organization']
    print(f"  [{article['domain']:15}] persons={persons}  orgs={orgs}")

Processed 10 articles
  [tech           ] persons=['Elon Musk']  orgs=['Tesla', 'Gigafactory']
  [sports         ] persons=['Cristiano Ronaldo']  orgs=['Al Nassr', 'Al Hilal']
  [tech           ] persons=['Tim Cook']  orgs=['Apple', 'Apple']
  [science        ] persons=[]  orgs=['CERN', 'MIT', 'Stanford University']
  [finance        ] persons=['Jamie Dimon']  orgs=['JPMorgan Chase', 'Senate Banking Committee']
  [tech           ] persons=['Sam Altman']  orgs=['OpenAI', 'board', 'Microsoft']
  [entertainment  ] persons=['Steven Spielberg']  orgs=['American Film Institute', 'Universal Pictures']
  [climate        ] persons=['António Guterres']  orgs=['UN', 'Paris Agreement']
  [healthcare     ] persons=[]  orgs=['Mayo Clinic']
  [philanthropy   ] persons=['Jeff Bezos', 'Lauren Sánchez']  orgs=['Amazon', 'Obama Foundation']


## 7. Pre-computing Label Embeddings (Bi-Encoder Advantage)

The bi-encoder separates text and label encoding. This means you can:
1. Encode labels **once** with `model.encode_labels()`
2. Cache the embeddings
3. Run inference on millions of documents with **zero extra label encoding cost**

This is the key advantage: inference time is constant regardless of label count.

In [9]:
labels = ["person", "organization", "location", "date", "product", "award", "event"]

# Step 1: encode labels once
entity_embeddings = model.encode_labels(labels, batch_size=8)
print(f"✓ Label embeddings shape: {entity_embeddings.shape}")
print(f"  {len(labels)} labels × embedding_dim={entity_embeddings.shape[1]}")

Encoding labels:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding labels: 100%|██████████| 1/1 [00:01<00:00,  1.45s/it]

Encoding labels: 100%|██████████| 1/1 [00:01<00:00,  1.45s/it]

✓ Label embeddings shape: torch.Size([7, 384])
  7 labels × embedding_dim=384


In [10]:
# Step 2: run inference with pre-computed embeddings
texts = [a["text"] for a in articles[:3]]

results = model.batch_predict_with_embeds(texts, entity_embeddings, labels, threshold=0.3)

print(f"Results with pre-computed embeddings:")
for text, entities in zip(texts, results):
    print(f"  Text: {text[:60]}...")
    for e in entities:
        print(f"    [{e['label']:12}] {e['text']!r}")
    print()

Results with pre-computed embeddings:
  Text: Elon Musk announced that Tesla will open a new Gigafactory i...
    [person      ] 'Elon Musk'
    [organization] 'Tesla'
    [organization] 'Gigafactory'
    [location    ] 'Austin'
    [location    ] 'Texas'
    [date        ] 'March 22, 2024'

  Text: Cristiano Ronaldo scored a hat-trick for Al Nassr against Al...
    [person      ] 'Cristiano Ronaldo'
    [organization] 'Al Nassr'
    [organization] 'Al Hilal'
    [date        ] 'Saturday'
    [location    ] 'King Fahd Stadium'
    [location    ] 'Riyadh'
    [location    ] 'Saudi Arabia'

  Text: Tim Cook unveiled the new Apple Vision Pro at the Worldwide ...
    [person      ] 'Tim Cook'
    [product     ] 'Apple Vision Pro'
    [event       ] 'Worldwide Developers Conference'
    [location    ] 'Cupertino'
    [location    ] 'California'
    [date        ] 'June 5, 2023'
    [organization] 'Apple'



In [11]:
import time

texts_bench = [a["text"] for a in articles]

# Without pre-computed embeddings
t0 = time.time()
for _ in range(3):
    model.inference(texts_bench, labels, threshold=0.3)
t_direct = (time.time() - t0) / 3

# With pre-computed embeddings
embs = model.encode_labels(labels)
t0 = time.time()
for _ in range(3):
    model.batch_predict_with_embeds(texts_bench, embs, labels, threshold=0.3)
t_cached = (time.time() - t0) / 3

print(f"Direct inference:          {t_direct:.3f}s")
print(f"Pre-computed embeddings:   {t_cached:.3f}s")
print(f"Speedup: {t_direct / t_cached:.1f}×")

Encoding labels:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding labels: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]

Encoding labels: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]

Direct inference:          0.993s
Pre-computed embeddings:   0.890s
Speedup: 1.1×


## Summary

- GLiNER uses natural language labels — any English description works
- `predict_entities(text, labels, threshold)` → list of `{text, label, start, end, score}`
- Character offsets in `start`/`end` let you do precise text manipulation
- `model.inference(texts, labels)` for batch processing
- `model.encode_labels(labels)` → pre-compute for constant-time inference at scale
- `model.batch_predict_with_embeds(texts, embeddings, labels)` → fast batch inference